# VL-JEPA (v1) — Vision-Language JEPA · Inference

*Notebook last updated: 2026/07/01*

**VL-JEPA** brings the joint-embedding predictive idea to **vision + language**:
images and captions are encoded into a shared latent space so that matching
image/text pairs land close together. This notebook runs **inference** with a
compact reference implementation.

### What this notebook does
1. Build the VL-JEPA model from `vl_jepa/config.yaml`.
2. Optionally load a trained checkpoint (otherwise weights are random).
3. Encode a batch of images and captions into a shared latent space.
4. Compute an **image↔text similarity matrix** — the diagonal should dominate
   for a well-trained model.

> **Heads-up:** with random weights (no checkpoint) the similarity matrix is
> essentially noise. Train first (see the training notebook) to see the diagonal
> light up.


In [1]:
# Uncomment the sys.path line if running as a script outside the package dir.
import sys
import os
#sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

In [2]:
import argparse
import torch
import torch.nn.functional as F
import yaml

In [3]:
# Local VL-JEPA package: model builder, tokenizer, dummy data, and helpers.
from vl_jepa.vl_jepa import build_vl_jepa
from vl_jepa.language_encoder import build_tokenizer
from vl_jepa.dataset import make_dummy_pairs
from vl_jepa.utils import set_seed, print_model_summary

In [4]:
# PIL/torchvision are optional; without them we fall back to random tensors.
try:
    from PIL import Image
    import torchvision.transforms as T
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

In [5]:
# Standard ImageNet-style preprocessing: resize -> center-crop -> normalize.
def load_image(path: str, img_size: int = 224) -> torch.Tensor:
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    tf = T.Compose([
        T.Resize(img_size, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    return tf(Image.open(path).convert("RGB")).unsqueeze(0)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
#p = argparse.ArgumentParser()
#p.add_argument("--checkpoint", default=None)
#p.add_argument("--config", default="config.yaml")
#args = p.parse_args()

### Load config & build the model

In [8]:
with open("vl_jepa/config.yaml") as f:
    cfg = yaml.safe_load(f)

In [13]:
# Set model_id to a checkpoint path to load trained weights;
# leave as None to use randomly-initialised weights (demo only).
model_id = None    # before training
#model_id = "vljepa_model/checkpoint_epoch0003.pt"     # after training

In [10]:
# Shrink model for quick demo if no checkpoint provided, otherwise use config in yaml file
# skip it
"""
if model_id is None:
    cfg["model"].update({
        "vision_embed_dim": 192, "vision_depth": 3, "vision_num_heads": 3,
        "lang_embed_dim": 128, "lang_depth": 2, "lang_num_heads": 2,
        "pred_depth": 2, "pred_num_heads": 3,
        "vision_proj_dim": 64, "lang_proj_dim": 64, "latent_dim": 64,
    })
"""

'\nif model_id is None:\n    cfg["model"].update({\n        "vision_embed_dim": 192, "vision_depth": 3, "vision_num_heads": 3,\n        "lang_embed_dim": 128, "lang_depth": 2, "lang_num_heads": 2,\n        "pred_depth": 2, "pred_num_heads": 3,\n        "vision_proj_dim": 64, "lang_proj_dim": 64, "latent_dim": 64,\n    })\n'

In [11]:
# Instantiate the vision encoder, language encoder, and predictor from config.
model = build_vl_jepa(cfg)
print_model_summary(model)


  VL-JEPA Model Summary
  Total params   :  310,194,176
  Trainable      :  223,608,064
  Frozen (EMA)   :   86,586,112



In [14]:
if model_id:
    ckpt = torch.load(model_id, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    print(f"Loaded checkpoint: {model_id}")
else:
    print("No checkpoint provided — using randomly initialised weights.")

No checkpoint provided — using randomly initialised weights.


In [15]:
model.to(device).eval()
tokenizer = build_tokenizer(cfg["model"]["vocab_size"], cfg["model"]["max_seq_len"])

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

### Prepare inputs and encode

`make_dummy_pairs` fabricates simple synthetic image/caption pairs so the notebook runs end-to-end with no dataset. Swap in real images for meaningful similarities.

In [16]:
# ── Synthetic images (use real images if available) ────────────────────
pairs = make_dummy_pairs(n=4, img_size=cfg["data"]["image_size"])
images = []
for pair in pairs:
    if HAS_PIL:
        images.append(load_image(pair.image_path, cfg["data"]["image_size"]))
    else:
        images.append(torch.randn(1, 3, cfg["data"]["image_size"], cfg["data"]["image_size"]))
images = torch.cat(images).to(device)

In [17]:
pairs[0].image_path

'./img_0000.jpg'

In [18]:
# ── Captions ───────────────────────────────────────────────────────────
captions = [
    "a dog running on the beach",
    "a cat sitting on a windowsill",
    "mountains covered in snow",
    "a cup of coffee on a wooden table",
]
token_ids, attn_mask = tokenizer.batch_encode(captions)
token_ids  = token_ids.to(device)
attn_mask  = attn_mask.to(device)

In [19]:
# Three views of the same shared space: image-only, text-only, and fused.
# Normalize so the dot product equals cosine similarity.
# ── Encode ────────────────────────────────────────────────────────────
with torch.no_grad():
    image_embs = model.encode_image(images)                     # (4, latent_dim)
    text_embs  = model.encode_text(token_ids, attn_mask)        # (4, latent_dim)
    mm_embs    = model.encode_multimodal(images, token_ids, attn_mask)  # (4, latent_dim)

image_embs = F.normalize(image_embs, dim=-1)
text_embs  = F.normalize(text_embs,  dim=-1)

sim = image_embs @ text_embs.T   # (4, 4)

In [20]:
# Row i = image i vs. every caption. For a trained model, entry [i,i]
# (matching pair) should be the largest in its row.
print("\n── Image-Text Similarity Matrix ──")
print("         ", "  ".join(f"T{i}" for i in range(len(captions))))
for i in range(len(images)):
    row = "  ".join(f"{sim[i, j].item():+.3f}" for j in range(len(captions)))
    print(f"  Image {i}: {row}")


── Image-Text Similarity Matrix ──
          T0  T1  T2  T3
  Image 0: -0.015  +0.022  -0.031  -0.035
  Image 1: -0.051  +0.020  -0.053  -0.028
  Image 2: -0.043  +0.005  -0.048  -0.027
  Image 3: -0.031  -0.018  -0.043  -0.060


In [21]:
print("\n── Embedding shapes ──")
print(f"  image_embs : {tuple(image_embs.shape)}")
print(f"  text_embs  : {tuple(text_embs.shape)}")
print(f"  mm_embs    : {tuple(mm_embs.shape)}")


── Embedding shapes ──
  image_embs : (4, 256)
  text_embs  : (4, 256)
  mm_embs    : (4, 256)


In [22]:
# Diagonal should be highest for a trained model
diagonal_mean = sim.diag().mean().item()
print(f"\n  Diagonal mean similarity: {diagonal_mean:.4f}")
print("  (expect ~1.0 for a well-trained model, ~0.0 for random weights)")


  Diagonal mean similarity: -0.0260
  (expect ~1.0 for a well-trained model, ~0.0 for random weights)
